# Pharmacy Desert Risk Modeling Using Machine Learning

# Week 2 – Data Collection and Initial Cleaning



## Objective

The objective of this notebook is to collect and prepare the main datasets required for pharmacy desert risk prediction. This includes CMS Medicare Part D pharmacy data, County Health Rankings data, and USDA ERS economic data. The cleaned outputs from this notebook will be used in later notebooks for merging, feature engineering, and modeling.

In [2]:
import os
import glob
import requests
import pandas as pd
import duckdb

## 1. CMS Medicare Part D Data Collection

This section collects CMS Medicare Part D pharmacy utilization data. The dataset includes pharmacy-related information such as claims, drug cost, fills, days supply, beneficiaries, and ZIP codes. Since the data is large, DuckDB is used for efficient processing and conversion.

## Pharmacy Datasets

CMS API Data, Link:https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider/data/2013
Data availabile from 2013-2023, data size of 8,294,940 observations

In [3]:
# Direct CMS CSV download URLs
files = {
    2013: "https://data.cms.gov/sites/default/files/2024-05/0d86f11e-1b96-426d-8f4f-f175184f9bcd/MUP_DPR_RY25_P04_V20_DY13_NPI.csv",
    2014: "https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY14_NPI.csv",
    2015: "https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY15_NPI.csv",
    2016: "https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY16_NPI.csv",
    2017: "https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY17_NPI.csv",
    2018: "https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY18_NPI.csv",
    2019: "https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY19_NPI.csv",
    2020: "https://data.cms.gov/sites/default/files/2022-07/27dd79c1-2fe9-4596-8b55-bfeaa2cd4ca8/MUP_DPR_RY22_P04_V10_DY20_NPI.csv",
    2021: "https://data.cms.gov/sites/default/files/2023-04/654a5915-691f-4d49-a49c-d0988fb56f86/MUP_DPR_RY23_P04_V10_DY21_NPI.csv",
    2022: "https://data.cms.gov/sites/default/files/2025-01/3f0ed44a-e1a1-4f51-b1c1-9f31f7e2db8f/MUP_DPR_RY25_P04_V20_DY22_NPI.csv",
    2023: "https://data.cms.gov/sites/default/files/2025-01/7d5436ab-6c64-4f84-b0f1-0d8df8d1ef7f/MUP_DPR_RY25_P04_V20_DY23_NPI.csv"
}

output_dir = "cms_partd_csvs"
os.makedirs(output_dir, exist_ok=True)

for year, url in files.items():

    print(f"\nDownloading {year}")
    print(url)

    try:
        response = requests.get(url, stream=True, timeout=300)
        response.raise_for_status()

        file_path = os.path.join(
            output_dir,
            f"medicare_partd_{year}.csv"
        )

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        print(f"Saved -> {file_path}")

    except Exception as e:
        print(f"Failed for {year}: {e}")


https://data.cms.gov/sites/default/files/2024-05/0d86f11e-1b96-426d-8f4f-f175184f9bcd/MUP_DPR_RY25_P04_V20_DY13_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2013.csv

https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY14_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2014.csv

https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY15_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2015.csv

https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY16_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2016.csv

https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY17_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2017.csv

https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY18_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2018.csv

https://data.cms.gov/sites/default/files/2021-08/MUP_DPR_RY21_P04_V10_DY19_NPI.csv
Saved -> cms_partd_csvs/medicare_partd_2019.csv

https://data.cms.gov/sites/default/fil

In [4]:
# Checking the correct data file downloads
for year, url in files.items():
    r = requests.get(url, stream=True)
    content_type = r.headers.get("Content-Type")
    print(year, content_type)

2013 text/html
2014 text/csv
2015 text/csv
2016 text/csv
2017 text/html
2018 text/csv
2019 text/csv
2020 text/csv
2021 text/csv
2022 text/html
2023 text/html


In [5]:
# Removing bad html files
bad_years = [2013, 2017, 2022, 2023]

for year in bad_years:
    path = f"cms_partd_csvs/medicare_partd_{year}.csv"
    if os.path.exists(path):
        os.remove(path)
        print("Removed", path)

Removed cms_partd_csvs/medicare_partd_2013.csv
Removed cms_partd_csvs/medicare_partd_2017.csv
Removed cms_partd_csvs/medicare_partd_2022.csv
Removed cms_partd_csvs/medicare_partd_2023.csv


In [6]:
dataset_ids = {
    2013: "93645ea4-1a3f-4444-95c5-c109e6a4b267",
    2017: "dc524dbb-6115-48c9-a4dc-aa4fb0d21b69",
    2022: "bed99012-c527-4d9d-92ea-67ec2510abea",
    2023: "6428fad7-4fb1-454c-93a4-772934d73922"
}

import requests

for year, uuid in dataset_ids.items():

    url = f"https://data.cms.gov/data-api/v1/dataset/{uuid}/data?size=2000000"

    r = requests.get(url).json()

    print(year, len(r))

2013 6500
2017 6500
2022 6500
2023 6500


In [7]:
# Dataset UUIDs
dataset_ids = {
    2013: "93645ea4-1a3f-4444-95c5-c109e6a4b267",
    2017: "dc524dbb-6115-48c9-a4dc-aa4fb0d21b69",
    2022: "bed99012-c527-4d9d-92ea-67ec2510abea",
    2023: "6428fad7-4fb1-454c-93a4-772934d73922"
}

# Output folder
os.makedirs("cms_partd_output", exist_ok=True)


# Downloader function
def download_year(year, uuid):

    print(f"\nStarting {year}...")

    offset = 0
    size = 6500
    first_write = True

    output_file = f"cms_partd_output/medicare_partd_{year}.csv"

    # removing old file if exists (optional safety)
    if os.path.exists(output_file):
        os.remove(output_file)

    while True:

        url = f"https://data.cms.gov/data-api/v1/dataset/{uuid}/data"

        params = {
            "size": size,
            "offset": offset
        }

        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()

        except Exception as e:
            print(f"Error at offset {offset}: {e}")
            break

        if not data:
            print("No more data, stopping.")
            break

        df = pd.DataFrame(data)

        # appending to CSV
        df.to_csv(
            output_file,
            mode="a",
            index=False,
            header=first_write
        )

        first_write = False

        offset += size

        print(f"{year} -> rows downloaded: {offset}")

        # stopping condition: last page
        if len(data) < size:
            print("Final page reached.")
            break

    print(f"Finished {year} -> saved to {output_file}")


# Running all years
for year, uuid in dataset_ids.items():
    download_year(year, uuid)


Starting 2013...
2013 -> rows downloaded: 6500
2013 -> rows downloaded: 13000
2013 -> rows downloaded: 19500
2013 -> rows downloaded: 26000
2013 -> rows downloaded: 32500
2013 -> rows downloaded: 39000
2013 -> rows downloaded: 45500
2013 -> rows downloaded: 52000
2013 -> rows downloaded: 58500
2013 -> rows downloaded: 65000
2013 -> rows downloaded: 71500
2013 -> rows downloaded: 78000
2013 -> rows downloaded: 84500
2013 -> rows downloaded: 91000
2013 -> rows downloaded: 97500
2013 -> rows downloaded: 104000
2013 -> rows downloaded: 110500
2013 -> rows downloaded: 117000
2013 -> rows downloaded: 123500
2013 -> rows downloaded: 130000
2013 -> rows downloaded: 136500
2013 -> rows downloaded: 143000
2013 -> rows downloaded: 149500
2013 -> rows downloaded: 156000
2013 -> rows downloaded: 162500
2013 -> rows downloaded: 169000
2013 -> rows downloaded: 175500
2013 -> rows downloaded: 182000
2013 -> rows downloaded: 188500
2013 -> rows downloaded: 195000
2013 -> rows downloaded: 201500
2013 -

In [8]:
CSV_FOLDER = "cms_partd_csvs"
API_FOLDER = "cms_partd_output"

OUT_FILE = "cms_merged_output/cms_partd_2013_2023.csv"

os.makedirs("cms_merged_output", exist_ok=True)

# removing old output if exists
if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)


def safe_read_csv(file_path):

    encodings = ["utf-8", "latin1", "ISO-8859-1", "cp1252"]

    for enc in encodings:
        try:
            return pd.read_csv(file_path, encoding=enc, low_memory=False)
        except:
            continue

    return pd.read_csv(file_path, encoding="latin1", errors="replace", low_memory=False)


def process_folder(folder, source_label):

    files = sorted(glob.glob(os.path.join(folder, "*.csv")))

    for f in files:

        print("Processing:", f)

        # reading file safely
        df = safe_read_csv(f)

        # metadata
        df["source"] = source_label
        df["file_origin"] = os.path.basename(f)

        # APPEND DIRECTLY TO DISK (no RAM buildup)
        df.to_csv(
            OUT_FILE,
            mode="a",
            index=False,
            header=not os.path.exists(OUT_FILE)
        )

        print("written:", len(df))


# RUN STREAMING PIPELINE
process_folder(CSV_FOLDER, "static_download")
process_folder(API_FOLDER, "api_download")

print("\nDONE → merged file written safely without RAM crash")

Processing: cms_partd_csvs/medicare_partd_2014.csv
written: 1072978
Processing: cms_partd_csvs/medicare_partd_2015.csv
written: 1102253
Processing: cms_partd_csvs/medicare_partd_2016.csv
written: 1131550
Processing: cms_partd_csvs/medicare_partd_2018.csv
written: 1204935
Processing: cms_partd_csvs/medicare_partd_2019.csv
written: 1240595
Processing: cms_partd_csvs/medicare_partd_2020.csv
written: 1255175
Processing: cms_partd_csvs/medicare_partd_2021.csv
written: 1287454
Processing: cms_partd_output/medicare_partd_2013.csv
written: 1049299
Processing: cms_partd_output/medicare_partd_2017.csv
written: 1162898
Processing: cms_partd_output/medicare_partd_2022.csv
written: 1332309
Processing: cms_partd_output/medicare_partd_2023.csv
written: 1380665

DONE → merged file written safely without RAM crash


In [9]:
# Loading Merged Data Using Duck Database
con = duckdb.connect()
# Getting Overview
df = con.execute("""
    SELECT *
    FROM read_csv_auto('cms_merged_output/cms_partd_2013_2023.csv')
    LIMIT 5
""").df()
df.head()

,PRSCRBR_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_MI,Prscrbr_Crdntls,Prscrbr_Gndr,Prscrbr_Ent_Cd,Prscrbr_St1,Prscrbr_St2,Prscrbr_City,...,Bene_Race_Black_Cnt,Bene_Race_Api_Cnt,Bene_Race_Hspnc_Cnt,Bene_Race_Natind_Cnt,Bene_Race_Othr_Cnt,Bene_Dual_Cnt,Bene_Ndual_Cnt,Bene_Avg_Risk_Scre,source,file_origin
0,1003000126,Enkeshafi,Ardalan,None,M.D.,M,I,900 Seton Dr,None,Cumberland,...,NaN,NaN,NaN,0.0,NaN,94.0,144.0,2.026587,static_download,medicare_partd_2014.csv
1,1003000142,Khalil,Rashid,None,M.D.,M,I,4126 N Holland Sylvania Rd,Suite 220,Toledo,...,44.0,0.0,NaN,0.0,NaN,82.0,64.0,1.653841,static_download,medicare_partd_2014.csv
2,1003000159,Voges,Marsha,S,FNP,F,I,4115 Dorchester Road,Concentra Medical Center,Charleston,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.360200,static_download,medicare_partd_2014.csv
3,1003000167,Escobar,Julio,E,DDS,M,I,5 Pine Cone Rd,None,Dayton,...,NaN,0.0,NaN,0.0,0.0,NaN,NaN,1.336192,static_download,medicare_partd_2014.csv
4,1003000282,Blakemore,Rosie,K,FNP,F,I,Tennessee Prison For Women,3881 Stewarts Lane,Nashville,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.872500,static_download,medicare_partd_2014.csv


In [10]:
# Getting Data Columns
schema = con.execute("""
    DESCRIBE SELECT *
    FROM read_csv_auto('cms_merged_output/cms_partd_2013_2023.csv')
""").df()
print(schema['column_name'].to_list())

['PRSCRBR_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_MI', 'Prscrbr_Crdntls', 'Prscrbr_Gndr', 'Prscrbr_Ent_Cd', 'Prscrbr_St1', 'Prscrbr_St2', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_zip5', 'Prscrbr_RUCA', 'Prscrbr_RUCA_Desc', 'Prscrbr_Cntry', 'Prscrbr_Type', 'Prscrbr_Type_src', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst', 'Tot_Day_Suply', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes', 'Brnd_Sprsn_Flag', 'Brnd_Tot_Clms', 'Brnd_Tot_Drug_Cst', 'Gnrc_Sprsn_Flag', 'Gnrc_Tot_Clms', 'Gnrc_Tot_Drug_Cst', 'Othr_Sprsn_Flag', 'Othr_Tot_Clms', 'Othr_Tot_Drug_Cst', 'MAPD_Sprsn_Flag', 'MAPD_Tot_Clms', 'MAPD_Tot_Drug_Cst', 'PDP_Sprsn_Flag', 'PDP_Tot_Clms', 'PDP_Tot_Drug_Cst', 'LIS_Sprsn_Flag', 'LIS_Tot_Clms', 'LIS_Drug_Cst', 'NonLIS_Sprsn_Flag', 'NonLIS_Tot_Clms', 'NonLIS_Drug_Cst', 'Opioid_Tot_Clms', 'Opioid_Tot_Drug_Cst', 'Opioid_Tot_Suply

In [11]:
# Converting to more workable format using Duck Database
con = duckdb.connect()
con.execute("""
    COPY (
        SELECT *
        FROM read_csv_auto(
            'cms_merged_output/cms_partd_2013_2023.csv',
            all_varchar=true,
            ignore_errors=true
        )
    )
    TO 'cms_partd.parquet'
    (FORMAT PARQUET);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
# Data size
con = duckdb.connect()

row_count = con.execute("""
    SELECT COUNT(*)
    FROM read_parquet('cms_partd.parquet')
""").fetchone()[0]

print("Rows:", row_count)

Rows: 8294940


CMS Medicare Part D data was downloaded and saved for later county-level aggregation and feature engineering.

## 2. County Health Rankings Data

This section processes County Health Rankings data from 2010 to 2023. These files provide county-level health outcome and health factor ranks, which help represent health conditions related to pharmacy access risk.

County Health Ranking Data, Link: https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation/national-data-documentation-2010-2023
Data availabile from 2010-2023, data size of 44,216 observations

In [19]:
# Importing Data
import pandas as pd
# File Path
file_path = "/content/CountyHealthRanking2010-2023.xlsx"
# Getting Years to extract data
years = ["2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]
# Storage Object
all_data = []
# Looping to get data
for year in years:
    print(f"Loading {year}")

    df = pd.read_excel(
        file_path,
        sheet_name=year,
        engine="openpyxl",
        header=1
    )

    # Removing first row
    df = df.iloc[1:].reset_index(drop=True)

    # Keeping only A, B, C, E, G
    df = df.iloc[:, [0, 1, 2, 4, 6]]

    df.columns = [
        "FIPS",
        "State",
        "County",
        "Health_Outcome_Rank",
        "Health_Factor_Rank"
    ]

    df["year"] = int(year)

    all_data.append(df)

    print("  -> shape:", df.shape)

# Combining Datasets
county_health_rank_final = pd.concat(all_data, ignore_index=True)

# Data Overview
print("\nFINAL SHAPE:", county_health_rank_final.shape)
county_health_rank_final.head()

Loading 2010
  -> shape: (3140, 6)
Loading 2011
  -> shape: (3140, 6)
Loading 2012
  -> shape: (3140, 6)
Loading 2013
  -> shape: (3191, 6)
Loading 2014
  -> shape: (3140, 6)
Loading 2015
  -> shape: (3140, 6)
Loading 2016
  -> shape: (3140, 6)
Loading 2017
  -> shape: (3135, 6)
Loading 2018
  -> shape: (3141, 6)
Loading 2019
  -> shape: (3141, 6)
Loading 2020
  -> shape: (3192, 6)
Loading 2021
  -> shape: (3192, 6)
Loading 2022
  -> shape: (3192, 6)
Loading 2023
  -> shape: (3192, 6)

FINAL SHAPE: (44216, 6)


,FIPS,State,County,Health_Outcome_Rank,Health_Factor_Rank,year
0,1003,Alabama,Baldwin,3,3,2010
1,1005,Alabama,Barbour,45,54,2010
2,1007,Alabama,Bibb,51,19,2010
3,1009,Alabama,Blount,10,9,2010
4,1011,Alabama,Bullock,63,62,2010


In [20]:
# Saving Data
county_health_rank_final.to_csv("county_health_rank_final.csv")

## 3. USDA ERS Economic Data

This section processes USDA ERS county-level economic data. The main economic variables used include employment, labor force, and unemployment rate. These variables help capture economic conditions that may influence pharmacy access.

USDA Economic Research Service (ERS) Data, Link: https://www.ers.usda.gov/data-products/county-level-data-sets/county-level-data-sets-download-data
Data availabile from 2000-2023, data size of 329,725 observations

In [21]:
# USDA ERS CSV URL
USDA_ERS_url = "https://www.ers.usda.gov/media/5497/unemployment-and-median-household-income-for-the-united-states-states-and-counties-2000-23.csv?v=53655"
# Loading directly from URL
USDA_ERS_df = pd.read_csv(USDA_ERS_url)
# Data Overview
USDA_ERS_df.head()

,FIPS_Code,State,Area_Name,Attribute,Value
0,0,US,United States,Civilian_labor_force_2000,142601576.0
1,0,US,United States,Employed_2000,136904853.0
2,0,US,United States,Unemployed_2000,5696723.0
3,0,US,United States,Unemployment_rate_2000,4.0
4,0,US,United States,Civilian_labor_force_2001,143786537.0


In [22]:
# Data Overview
USDA_ERS_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 329726 entries, 0 to 329725
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   FIPS_Code  329726 non-null  int64  
 1   State      329726 non-null  object 
 2   Area_Name  329726 non-null  object 
 3   Attribute  329726 non-null  object 
 4   Value      329726 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 12.6+ MB


In [23]:
# Data Cleaning
# Removing national-level data
USDA_ERS_df = USDA_ERS_df[USDA_ERS_df["Area_Name"] != "United States"]

# Converting to wide format
USDA_ERS_df_wide = USDA_ERS_df.pivot_table(
    index=["FIPS_Code", "State", "Area_Name"],
    columns="Attribute",
    values="Value",
    aggfunc="first"
).reset_index()


USDA_ERS_df_wide.head()


Attribute,FIPS_Code,State,Area_Name,Civilian_labor_force_2000,Civilian_labor_force_2001,Civilian_labor_force_2002,Civilian_labor_force_2003,Civilian_labor_force_2004,Civilian_labor_force_2005,Civilian_labor_force_2006,...,Unemployment_rate_2015,Unemployment_rate_2016,Unemployment_rate_2017,Unemployment_rate_2018,Unemployment_rate_2019,Unemployment_rate_2020,Unemployment_rate_2021,Unemployment_rate_2022,Unemployment_rate_2023,Urban_Influence_Code_2013
0,1000,AL,Alabama,2147173.0,2128027.0,2112621.0,2128668.0,2138306.0,2140356.0,2170007.0,...,6.1,5.9,4.5,3.9,3.2,6.4,3.4,2.5,2.5,NaN
1,1001,AL,"Autauga County, AL",21861.0,22081.0,22161.0,22695.0,23241.0,23887.0,24425.0,...,5.2,5.1,4.0,3.6,2.9,5.3,2.8,2.2,2.2,2.0
2,1003,AL,"Baldwin County, AL",69979.0,69569.0,69379.0,72598.0,74843.0,76608.0,79806.0,...,5.6,5.4,4.2,3.6,2.9,6.1,2.9,2.3,2.3,2.0
3,1005,AL,"Barbour County, AL",11449.0,11324.0,11006.0,11019.0,10639.0,10730.0,10713.0,...,8.9,8.4,6.0,5.1,4.0,7.7,5.5,4.0,4.4,6.0
4,1007,AL,"Bibb County, AL",8623.0,9134.0,8961.0,8871.0,8851.0,8837.0,8858.0,...,6.7,6.5,4.5,4.0,3.2,7.2,3.4,2.4,2.5,1.0


In [24]:
# Fixing Columns
print(USDA_ERS_df_wide.columns.to_list())

# Identifier columns
id_vars = ["FIPS_Code", "State", "Area_Name"]

# Melting everything else
USDA_ERS_df_long = USDA_ERS_df_wide.melt(
    id_vars=id_vars,
    var_name="variable",
    value_name="value"
)

# Extracting attribute and year
USDA_ERS_df_long[["attribute", "year"]] = USDA_ERS_df_long["variable"].str.extract(
    r"(.+)_(\d{4})$"
)

# Keeping only rows with valid years
USDA_ERS_df_long = USDA_ERS_df_long.dropna(subset=["year"])

# Converting year to integer
USDA_ERS_df_long["year"] = USDA_ERS_df_long["year"].astype(int)

# Pivot attributes into columns
USDA_ERS_df_tidy = USDA_ERS_df_long.pivot_table(
    index=["FIPS_Code", "State", "Area_Name", "year"],
    columns="attribute",
    values="value",
    aggfunc="first"
).reset_index()

# Removing column index name
USDA_ERS_df_tidy.columns.name = None

# Reanaming Map
rename_map = {
    "Civilian_labor_force": "labor_force",
    "Employed": "employed",
    "Unemployed": "unemployed",
    "Unemployment_rate": "unemployment_rate",
    "Median_Household_Income": "median_household_income",
    "Med_HH_Income_Percent_of_State_Total": "income_pct_state",
    "Metro": "metro",
    "Rural_Urban_Continuum_Code": "rural_urban_code",
    "Urban_Influence_Code": "urban_influence_code"
}

USDA_ERS_df_tidy = USDA_ERS_df_tidy.rename(columns=rename_map)


['FIPS_Code', 'State', 'Area_Name', 'Civilian_labor_force_2000', 'Civilian_labor_force_2001', 'Civilian_labor_force_2002', 'Civilian_labor_force_2003', 'Civilian_labor_force_2004', 'Civilian_labor_force_2005', 'Civilian_labor_force_2006', 'Civilian_labor_force_2007', 'Civilian_labor_force_2008', 'Civilian_labor_force_2009', 'Civilian_labor_force_2010', 'Civilian_labor_force_2011', 'Civilian_labor_force_2012', 'Civilian_labor_force_2013', 'Civilian_labor_force_2014', 'Civilian_labor_force_2015', 'Civilian_labor_force_2016', 'Civilian_labor_force_2017', 'Civilian_labor_force_2018', 'Civilian_labor_force_2019', 'Civilian_labor_force_2020', 'Civilian_labor_force_2021', 'Civilian_labor_force_2022', 'Civilian_labor_force_2023', 'Employed_2000', 'Employed_2001', 'Employed_2002', 'Employed_2003', 'Employed_2004', 'Employed_2005', 'Employed_2006', 'Employed_2007', 'Employed_2008', 'Employed_2009', 'Employed_2010', 'Employed_2011', 'Employed_2012', 'Employed_2013', 'Employed_2014', 'Employed_201

In [25]:
# Data Overview
USDA_ERS_df_tidy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78413 entries, 0 to 78412
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   FIPS_Code                78413 non-null  int64  
 1   State                    78413 non-null  object 
 2   Area_Name                78413 non-null  object 
 3   year                     78413 non-null  int64  
 4   labor_force              78395 non-null  float64
 5   employed                 78395 non-null  float64
 6   income_pct_state         3194 non-null   float64
 7   median_household_income  3194 non-null   float64
 8   metro                    3221 non-null   float64
 9   rural_urban_code         3221 non-null   float64
 10  unemployed               78395 non-null  float64
 11  unemployment_rate        78395 non-null  float64
 12  urban_influence_code     3219 non-null   float64
dtypes: float64(9), int64(2), object(2)
memory usage: 7.8+ MB


In [26]:
# Saving Data
USDA_ERS_df_tidy.to_csv("USDA_ERS_df_tidy.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Final Output Summary

In this notebook, the main datasets required for the pharmacy desert risk prediction project were collected and initially cleaned.

The completed steps include:

• CMS Medicare Part D data collection and storage  
• County Health Rankings data cleaning  
• USDA ERS economic data cleaning  
• Missing value handling  
• Cleaned output files saved for later merging and feature engineering  

The next step is to merge these datasets at the county-year level and create key features such as PAP, TDF, spatial lag features, and the final Desert Formation Risk label.